# A/B Test Design

07에서 정의한 retention, cx 개선 대상을 기반으로 실제 서비스에서 검증할 A/B test를 설계한다.

- retention target: 첫 주문금액 q3, q4 고객
- cx target: 예상 배송일 기준 4일 이상 지연 고객
- 목적: intervention의 효과를 검증할 실험 구조와 필요한 표본 규모를 정의한다.

> Olist는 treatment/control이 존재하지 않는 관찰데이터이므로 실제 실험 결과가 아닌 실험 설계와 power analysis까지 수행한다.

## 1. Experiment Setup

07 분석에서 정의한 target과 baseline을 기반으로 retention, cx intervention의 효과를 검증할 실험을 설계한다.

| Objective | Target | Primary KPI | Baseline |
| --- | --- | --- | ---: |
| Retention | 첫 주문금액 q3, q4 고객 | 90d repurchase rate | 2.12% |
| CX | 예상 배송일 기준 4일 이상 지연 | low review rate | 75.02% |

### Common Design

- target 고객을 treatment / control에 무작위 배정한다.
- 기본 배정 비율은 1:1로 설정한다.
- treatment와 control은 동일한 관찰 기간을 적용한다.
- 유의수준은 0.05, 검정력은 0.80을 기준으로 한다.
- 현재 가용 sample size를 기준으로 검출 가능한 효과 크기를 계산한다.
- 실제 운영에서는 비즈니스적으로 의미 있는 MDE를 먼저 정의하고 필요한 sample size를 산정한다.
- 결과는 %p 변화, 상대 lift, 신뢰구간, p-value를 함께 확인한다.
- cohort별 성과 차이가 확인됐으므로 실험 기간과 시기 효과를 함께 고려한다.
- treatment / control의 표본 비율과 category, cohort 분포를 확인해 randomization이 정상적으로 이루어졌는지 점검한다.

## 2. Retention Experiment

q3, q4 첫 구매 고객을 대상으로 category 기반 재구매 CRM이 90일 재구매율을 높이는지 검증한다.

### Hypothesis

- H0: treatment와 control의 90일 재구매율에 차이가 없다.
- H1: treatment와 control의 90일 재구매율에 차이가 있다.
- treatment의 재구매율이 control보다 높을 때 개선 효과로 해석한다.

### Design

- target: 첫 주문금액 q3, q4 고객
- randomization unit: `customer_unique_id`
- control: 추가 재구매 유도 메시지 없이 기존 구매 경험을 유지한다.
- treatment: 첫 구매 category를 기반으로 연관 상품을 추천하는 재구매 유도 푸시 메시지를 제공한다.
- intervention timing: 첫 구매 이후 일정 시점에 1회 발송한다.
- primary KPI: 90d repurchase rate
- baseline: 2.12%
- observation period: 첫 구매일 기준 90일
- category와 cohort는 실험 결과 해석 시 함께 확인한다.

### Detectable Effect

현재 q3, q4 target pool 46,878명을 향후 실험에서 모두 90일간 관찰할 수 있다고 가정하고, treatment / control에 1:1로 배정했을 때 유의수준 0.05, 검정력 0.80에서 검출 가능한 최소 재구매율 차이를 계산한다.

In [2]:
# 현재 target 규모에서 검출 가능한 최소 효과 계산
from statsmodels.stats.power import NormalIndPower
import math

baseline = 0.0212
total_sample = 46878
group_sample = total_sample // 2

power_analysis = NormalIndPower()

effect_size = power_analysis.solve_power(
	effect_size=None,
	nobs1=group_sample,
	alpha=0.05,
	power=0.80,
	ratio=1,
	alternative='two-sided'
)

baseline_angle = 2 * math.asin(math.sqrt(baseline))
target_rate = math.sin((baseline_angle + effect_size) / 2) ** 2

print('group별 sample size:', group_sample)
print('baseline:', f'{baseline * 100:.2f}%')
print('detectable target rate:', f'{target_rate * 100:.2f}%')
print('detectable %p change:', f'+{(target_rate - baseline) * 100:.2f}%p')
print('relative lift:', f'+{(target_rate - baseline) / baseline * 100:.1f}%')

group별 sample size: 23439
baseline: 2.12%
detectable target rate: 2.51%
detectable %p change: +0.39%p
relative lift: +18.3%


### Decision Rule

- treatment의 90일 재구매율이 control보다 높은지 확인한다.
- p-value가 0.05 미만이면 두 그룹의 차이가 통계적으로 유의하다고 판단한다.
- 재구매율이 얼마나 상승했는지는 %p 변화와 상대 lift로 함께 확인한다.
- 통계적으로 유의하더라도 실제 적용 여부는 CRM 비용과 기대 수익을 함께 고려해 결정한다.
- 현재 sample size에서는 약 +0.39%p 이상의 차이를 비교적 안정적으로 검출할 수 있다.

## 3. CX Experiment

예상 배송일보다 4일 이상 지연된 고객을 대상으로 선제 안내 및 service recovery가 저리뷰 발생을 줄이는지 검증한다.

### Hypothesis

- H0: treatment와 control의 저리뷰율에 차이가 없다.
- H1: treatment와 control의 저리뷰율에 차이가 있다.
- treatment의 저리뷰율이 control보다 낮을 때 개선 효과로 해석한다.

### Design

- target: 예상 배송일 기준 4일 이상 지연 고객
- randomization unit: `customer_unique_id`
- control: 기존 배송 지연 대응을 유지한다.
- treatment: 4일 이상 지연 시 지연 사실과 예상 배송 정보를 선제 안내하고 보상 쿠폰을 제공한다.
- intervention timing: 예상 배송일 초과 4일째에 개입한다.
- primary KPI: low review rate
- secondary KPI: avg review score, review submission rate
- baseline: 75.02%
- observation period: 해당 주문의 리뷰 작성 또는 실험 종료 시점까지
- cohort는 실험 결과 해석 시 함께 확인한다.

### Detectable Effect

4일 이상 지연 고객 중 리뷰 확인이 가능한 4,404명을 treatment / control에 1:1로 배정한다고 가정하고, 유의수준 0.05, 검정력 0.80에서 검출 가능한 최소 저리뷰율 차이를 계산한다.

In [4]:
# 현재 분석 가능 규모에서 검출 가능한 최소 효과 계산
from statsmodels.stats.power import NormalIndPower
import math

baseline = 0.7502
total_sample = 4404
group_sample = total_sample // 2

power_analysis = NormalIndPower()

effect_size = power_analysis.solve_power(
	effect_size=None,
	nobs1=group_sample,
	alpha=0.05,
	power=0.80,
	ratio=1,
	alternative='two-sided'
)

baseline_angle = 2 * math.asin(math.sqrt(baseline))
target_rate = math.sin((baseline_angle - effect_size) / 2) ** 2

print('group별 sample size:', group_sample)
print('baseline:', f'{baseline * 100:.2f}%')
print('detectable target rate:', f'{target_rate * 100:.2f}%')
print('detectable %p change:', f'{(target_rate - baseline) * 100:.2f}%p')
print('relative reduction:', f'{(baseline - target_rate) / baseline * 100:.1f}%')

group별 sample size: 2202
baseline: 75.02%
detectable target rate: 71.28%
detectable %p change: -3.74%p
relative reduction: 5.0%


### Decision Rule

- treatment의 저리뷰율이 control보다 낮은지 확인한다.
- p-value가 0.05 미만이면 두 그룹의 차이가 통계적으로 유의하다고 판단한다.
- 저리뷰율이 얼마나 감소했는지는 %p 변화와 상대 감소율로 함께 확인한다.
- 통계적으로 유의하더라도 실제 적용 여부는 service recovery 비용과 cx 개선 효과를 함께 고려해 결정한다.
- 현재 분석 가능 sample size에서는 약 -3.74%p 이상의 차이를 비교적 안정적으로 검출할 수 있다.

## 4. Final Experiment Plan

| Objective | Target | Treatment | Primary KPI | Baseline | Detectable Effect |
| --- | --- | --- | --- | ---: | ---: |
| Retention | 첫 주문금액 q3, q4 고객 | category 기반 재구매 유도 푸시 | 90d repurchase rate | 2.12% | 약 +0.39%p |
| CX | 예상 배송일 기준 4일 이상 지연 고객 | 선제 배송 안내 + 보상 쿠폰 | low review rate | 75.02% | 약 -3.74%p |

### Limitations

- Olist에는 실제 treatment / control 배정 정보가 없어 실험 결과까지 검증할 수 없다.
- detectable effect는 historical target 규모를 가용 sample size로 가정해 계산한 값이며, 비즈니스 성공 기준을 의미하지 않는다.
- 실제 운영에서는 비용과 기대 효과를 기준으로 MDE를 정의하고, 무작위 실험을 통해 intervention의 실제 효과를 검증해야 한다.